In [7]:
import re
import shutil
import pdfplumber
import pandas as pd
from pathlib import Path

CARPETA_RECIBOS = Path("./Recibos")
CARPETA_LEIDOS = CARPETA_RECIBOS / "Leidos"
EXCEL_PATH = "precios_productos.xlsx"         # fichero maestro (se lee primero y se actualiza)


def _normalizar_fecha_valor(valor) -> str | None:
    """Convierte el índice/fecha leída del Excel al formato dd/mm/yyyy del ticket."""
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None
    if isinstance(valor, pd.Timestamp):
        return valor.strftime("%d/%m/%Y")
    s = str(valor).strip()
    m = re.match(r"(\d{4})-(\d{2})-(\d{2})", s)
    if m:
        y, mo, d = m.groups()
        return f"{d}/{mo}/{y}"
    return s


def clave_desde_indice_excel(fecha_raw) -> str | None:
    """Índice del Excel → etiqueta de fila (conserva sufijos '(2)', '(3)' si ya existían)."""
    if fecha_raw is None or (isinstance(fecha_raw, float) and pd.isna(fecha_raw)):
        return None
    if isinstance(fecha_raw, pd.Timestamp):
        return _normalizar_fecha_valor(fecha_raw)
    s = str(fecha_raw).strip()
    m_iso = re.match(r"^(\d{4})-(\d{2})-(\d{2})", s)
    if m_iso:
        y, mo, d = m_iso.groups()
        return f"{d}/{mo}/{y}"
    return s


def _base_fecha_desde_clave(clave: str) -> str:
    """Parte dd/mm/yyyy de una clave '12/05/2026' o '12/05/2026 (2)'."""
    m = re.match(r"^(\d{2}/\d{2}/\d{4})", str(clave).strip())
    return m.group(1) if m else str(clave).strip()


def clave_fila_para_nuevo_ticket(fecha_ticket: str, datos: dict) -> str:
    """
    Cada ticket nuevo = una fila distinta. Varios tickets el mismo día:
    primera fila 'dd/mm/yyyy', siguientes 'dd/mm/yyyy (2)', '(3)', …
    """
    base = _base_fecha_desde_clave(fecha_ticket)
    if base not in datos:
        return base
    n = 2
    while f"{base} ({n})" in datos:
        n += 1
    return f"{base} ({n})"


def normalizar_nombre_producto(nombre: str) -> str:
    """
    Unifica variantes casi idénticas del mismo producto para no duplicar columnas:
    espacios múltiples, espacios alrededor del guion, NBSP y guiones tipográficos.
    Ej.: "12 HUEVOS GRANDES -L" → misma clave que "12 HUEVOS GRANDES-L".
    """
    if nombre is None:
        return ""
    s = str(nombre).replace("\u00a0", " ").strip()
    s = re.sub(r"\s+", " ", s)
    for guion in ("–", "—", "−"):
        s = s.replace(guion, "-")
    s = re.sub(r"\s*-\s*", "-", s)
    return s.strip()


def cargar_excel_a_datos(excel_path: str) -> tuple[dict, int]:
    """
    Carga el maestro Excel a {clave_fila: {producto: precio}}.
    Devuelve (datos, n_filas_hoja) con n_filas_hoja = filas del índice del fichero (incl. vacías).
    """
    p = Path(excel_path)
    if not p.is_file():
        return {}, 0
    try:
        df = pd.read_excel(p, index_col=0)
    except Exception as e:
        print(f"⚠️  No se pudo leer el Excel existente ({e}). Se empieza vacío.")
        return {}, 0
    if df.empty:
        return {}, 0
    n_filas_hoja = len(df)
    datos: dict = {}
    for fecha_raw, row in df.iterrows():
        clave_guess = clave_desde_indice_excel(fecha_raw)
        if not clave_guess:
            continue
        productos = {}
        for col in df.columns:
            val = row[col]
            if pd.notna(val):
                try:
                    clave_prod = normalizar_nombre_producto(str(col))
                    if clave_prod:
                        productos[clave_prod] = float(val)
                except (TypeError, ValueError):
                    pass
        if not productos:
            continue
        if clave_guess not in datos:
            destino = clave_guess
        else:
            base = _base_fecha_desde_clave(clave_guess)
            destino = clave_fila_para_nuevo_ticket(base, datos)
        if destino not in datos:
            datos[destino] = {}
        datos[destino].update(productos)
    n_claves = len(datos)
    print(f"📊 Excel existente: {n_filas_hoja} fila(s) en hoja, {n_claves} ticket(s) con datos — {p.resolve()}")
    return datos, n_filas_hoja


def extraer_fecha(texto: str) -> str:
    """
    Devuelve la fecha en formato dd/mm/yyyy encontrada en el ticket.
    """
    m = re.search(r"(\d{2}/\d{2}/\d{4})", texto)
    if not m:
        raise ValueError("No se ha encontrado la fecha en el ticket.")
    return m.group(1)


def es_linea_producto(linea: str) -> bool:
    """
    Consideramos línea de producto si empieza por un entero (cantidad).
    Ejemplo: '8 CAVA SEMI SECO 3,20 25,60'
    """
    return bool(re.match(r"^\s*\d+\s+", linea))


def parsear_linea_producto(linea: str):
    """
    Devuelve (cantidad:int, descripcion:str, precio_unit:float)
    o None si la línea no se puede interpretar.

    Lógica:
    - Formato típico:
        '8 CAVA SEMI SECO 3,20 25,60'
        '1 PAN BLANCO FAMILIAR 1,18'
    - Si hay dos precios al final -> primero = P.Unit
    - Si hay un precio -> P.Unit = precio / cantidad
    """
    linea = linea.strip()
    # Coincide: cantidad, descripción y hasta dos números decimales al final
    m = re.match(
        r"^(\d+)\s+(.*?)(\d+,\d{2})(?:\s+(\d+,\d{2}))?$",
        linea
    )
    if not m:
        return None

    cantidad = int(m.group(1))
    descripcion = m.group(2).strip()
    num1 = m.group(3)
    num2 = m.group(4)  # puede ser None

    def to_float(s):
        return float(s.replace(".", "").replace(",", "."))

    if num2 is not None:
        # dos números: num1 = P.Unit, num2 = total
        precio_unit = to_float(num1)
    else:
        # un número: es el importe total -> calculamos P.Unit
        importe_total = to_float(num1)
        precio_unit = importe_total / cantidad if cantidad != 0 else importe_total

    return cantidad, descripcion, precio_unit


def leer_ticket(pdf_path: str):
    """
    Devuelve (fecha:str, dict_productos:{descripcion:precio_unit})
    para un único ticket en PDF.
    """
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[0]
        texto = page.extract_text()

    fecha = extraer_fecha(texto)

    productos = {}
    dentro_tabla = False

    for linea in texto.splitlines():
        linea = linea.strip()

        if "Descripción" in linea and "P. Unit" in linea:
            # A partir de aquí empiezan los productos
            dentro_tabla = True
            continue
        if "TOTAL (€)" in linea:
            # Fin de la tabla de productos
            break
        if not dentro_tabla or not linea:
            continue

        if es_linea_producto(linea):
            parsed = parsear_linea_producto(linea)
            if parsed is None:
                continue
            _, descripcion, precio_unit = parsed
            clave_prod = normalizar_nombre_producto(descripcion)
            if clave_prod:
                productos[clave_prod] = precio_unit

    return fecha, productos


def acumular_datos(datos_acumulados: dict, clave_fila: str, productos: dict):
    """
    Acumula los datos de productos en un diccionario en memoria.
    Estructura: {clave_fila: {producto: precio}} — clave_fila suele ser la fecha del ticket
    o 'fecha (2)' si hubo otro ticket el mismo día.

    Args:
        datos_acumulados: Diccionario donde se acumulan los datos
        clave_fila: Etiqueta de fila en el Excel (fecha o fecha + sufijo)
        productos: Diccionario con productos y precios
    """
    if clave_fila not in datos_acumulados:
        datos_acumulados[clave_fila] = {}

    # Misma clave canónica para variantes de nombre (espacios / guiones)
    norm = {
        normalizar_nombre_producto(k): v
        for k, v in productos.items()
        if normalizar_nombre_producto(k)
    }
    datos_acumulados[clave_fila].update(norm)


def guardar_excel_final(datos_acumulados: dict, excel_path: str) -> bool:
    """
    Guarda todos los datos acumulados en un Excel.
    Estructura: una fila por ticket (clave = fecha o fecha (2), …), productos en columnas.
    Devuelve True si se escribió el fichero correctamente.
    """
    excel_file = Path(excel_path)

    if not datos_acumulados:
        print("⚠️  No hay datos para guardar")
        return False

    # Obtener todas las fechas y todos los productos únicos
    todas_las_fechas = sorted(datos_acumulados.keys())
    todos_los_productos = set()
    for productos_dict in datos_acumulados.values():
        todos_los_productos.update(productos_dict.keys())
    todos_los_productos = sorted(todos_los_productos)

    # Crear DataFrame con fechas como índice (filas) y productos como columnas
    df = pd.DataFrame(index=todas_las_fechas, columns=todos_los_productos, dtype=float)

    # Llenar el DataFrame con los datos
    for fecha in todas_las_fechas:
        for producto, precio in datos_acumulados[fecha].items():
            df.loc[fecha, producto] = precio

    try:
        df.to_excel(excel_file, index=True, index_label="Fecha")
    except Exception as e:
        print(f"❌ Error al guardar Excel: {e}")
        return False

    print(f"✅ Excel guardado: {excel_file.resolve()}")
    print(f"   • Tickets (filas con datos): {len(df)}")
    print(f"   • Productos (columnas): {len(df.columns)}")
    return True


def mover_a_leidos(pdf_path: Path) -> bool:
    """Tras guardar el maestro, archiva el PDF en Recibos/Leidos/."""
    try:
        CARPETA_LEIDOS.mkdir(parents=True, exist_ok=True)
        destino = CARPETA_LEIDOS / pdf_path.name
        if destino.exists():
            contador = 1
            nombre_base = pdf_path.stem
            extension = pdf_path.suffix
            while destino.exists():
                destino = CARPETA_LEIDOS / f"{nombre_base}_{contador}{extension}"
                contador += 1
        shutil.move(str(pdf_path), str(destino))
        print(f"  📦 Movido a: {destino}")
        return True
    except Exception as e:
        print(f"  ❌ Error moviendo {pdf_path.name}: {e}")
        return False


def main():
    """Carga el Excel, fusiona PDFs de CARPETA_RECIBOS (varios tickets el mismo día → varias filas), guarda y archiva en Leidos."""

    if not CARPETA_RECIBOS.exists():
        print(f"❌ La carpeta '{CARPETA_RECIBOS}' no existe.")
        print("   Por favor, crea la carpeta y coloca los PDFs allí.")
        return

    datos_acumulados, filas_hoja_inicio = cargar_excel_a_datos(EXCEL_PATH)
    claves_inicio = len(datos_acumulados)

    vistos = set()
    pdfs = []
    for patron in ("*.pdf", "*.PDF"):
        for f in CARPETA_RECIBOS.glob(patron):
            if f.is_file() and f.parent == CARPETA_RECIBOS:
                r = f.resolve()
                if r not in vistos:
                    vistos.add(r)
                    pdfs.append(f)

    if not pdfs:
        print(f"\nℹ️  No se encontraron archivos PDF en '{CARPETA_RECIBOS}'")
        print("\n📋 Control de filas (sin cambios en el maestro):")
        print(f"   • Filas en la hoja Excel al inicio (índice): {filas_hoja_inicio}")
        print(f"   • Tickets con datos cargados del maestro: {claves_inicio}")
        return

    print(f"\n📄 Encontrados {len(pdfs)} archivo(s) PDF para procesar\n")

    pdfs_incluidos_en_esta_corrida = []
    procesados_fallidos = 0

    for pdf_path in sorted(pdfs, key=lambda x: x.name.lower()):
        print(f"\n{'='*60}")
        print(f"Archivo: {pdf_path.name}")
        print(f"{'='*60}")

        try:
            fecha, productos = leer_ticket(str(pdf_path))
            clave_fila = clave_fila_para_nuevo_ticket(fecha, datos_acumulados)
            print(f"Fecha ticket: {fecha}")
            print(f"Fila en el maestro: {clave_fila}")
            print(f"Productos encontrados: {len(productos)}")
            for d, p in list(productos.items())[:5]:
                print(f"  - {d}: {p:.2f} €")
            if len(productos) > 5:
                print(f"  ... y {len(productos) - 5} productos más")

            acumular_datos(datos_acumulados, clave_fila, productos)
            pdfs_incluidos_en_esta_corrida.append(pdf_path)
            print("  ✅ Datos fusionados en memoria (se guardará con el resto del maestro)")

        except Exception as e:
            print(f"  ❌ Error procesando {pdf_path.name}: {e}")
            procesados_fallidos += 1

    if not pdfs_incluidos_en_esta_corrida:
        print(f"\n⚠️  Ningún PDF se integró correctamente; el maestro en disco no se modifica.")
        print(f"\n{'='*60}")
        print("RESUMEN")
        print(f"{'='*60}")
        print("\n📋 Control de filas:")
        print(f"   • Filas en la hoja Excel al inicio (índice): {filas_hoja_inicio}")
        print(f"   • Tickets con datos al inicio: {claves_inicio}")
        print(f"   • PDFs leídos correctamente: 0")
        print(f"   • Tickets con datos al final (sin guardar): {len(datos_acumulados)}")
        print(f"❌ Fallidos: {procesados_fallidos}")
        print(f"📁 Total PDFs en carpeta: {len(pdfs)}")
        return

    print(f"\n{'='*60}")
    print("GUARDANDO DATOS EN EXCEL...")
    print(f"{'='*60}")
    guardado_ok = guardar_excel_final(datos_acumulados, EXCEL_PATH)

    movidos = 0
    if guardado_ok and pdfs_incluidos_en_esta_corrida:
        print(f"\n{'='*60}")
        print("ARCHIVANDO PDFs EN Leidos/ ...")
        print(f"{'='*60}")
        for pdf_path in pdfs_incluidos_en_esta_corrida:
            if pdf_path.is_file() and mover_a_leidos(pdf_path):
                movidos += 1
    elif not guardado_ok:
        print("\n⚠️  No se mueven PDFs a Leidos porque el guardado del Excel falló.")

    claves_fin = len(datos_acumulados)
    n_ok = len(pdfs_incluidos_en_esta_corrida)
    esperado = claves_inicio + n_ok
    coherente = claves_fin == esperado

    print(f"\n{'='*60}")
    print("RESUMEN")
    print(f"{'='*60}")
    print("\n📋 Control de filas / tickets:")
    print(f"   • Filas en la hoja Excel al inicio (índice del fichero): {filas_hoja_inicio}")
    print(f"   • Tickets con datos en memoria al inicio (tras cargar maestro): {claves_inicio}")
    print(f"   • PDFs leídos e integrados en esta ejecución: {n_ok}")
    print(f"   • Tickets con datos en memoria al final (coincide con filas del xlsx guardado): {claves_fin}")
    if coherente:
        print(f"   • Comprobación: {claves_inicio} + {n_ok} = {esperado}  →  ✅ coincide con {claves_fin}")
    else:
        print(f"   • Comprobación: {claves_inicio} + {n_ok} = {esperado}  →  ⚠️ NO coincide ({claves_fin}); revisa filas sin precios u hojas con índices duplicados")
    print(f"\n✅ PDFs integrados en el maestro: {n_ok}")
    print(f"📦 PDFs movidos a Leidos/: {movidos}")
    print(f"❌ Fallidos: {procesados_fallidos}")
    print(f"📁 Total PDFs en carpeta al inicio: {len(pdfs)}")


if __name__ == "__main__":
    main()



📄 Encontrados 174 archivo(s) PDF para procesar


Archivo: 20231125 Mercadona 102,29 €.pdf
Fecha ticket: 25/11/2023
Fila en el maestro: 25/11/2023
Productos encontrados: 39
  - CANÓNIGOS: 1.14 €
  - T.CHERRY 500 GR: 1.95 €
  - PAT. CLASSICAS: 1.75 €
  - CROQUETA JAMON: 1.25 €
  - TRIÁNGULOS DE MAÍZ: 2.05 €
  ... y 34 productos más
  ✅ Datos fusionados en memoria (se guardará con el resto del maestro)

Archivo: 20231202 Mercadona 61,98 €.pdf
Fecha ticket: 02/12/2023
Fila en el maestro: 02/12/2023
Productos encontrados: 27
  - 12 HUEVOS GRANDES-L: 2.20 €
  - YOGUR NATURAL X6: 1.05 €
  - YOGOMIX X6: 3.05 €
  - SALADITOS: 1.30 €
  - SUAVIZANTE AZUL: 1.85 €
  ... y 22 productos más
  ✅ Datos fusionados en memoria (se guardará con el resto del maestro)

Archivo: 20231209 Mercadona 7,92 €.pdf
Fecha ticket: 09/12/2023
Fila en el maestro: 09/12/2023
Productos encontrados: 3
  - CERVEZA CLASICA P-12: 3.72 €
  - SAL FINA: 0.35 €
  - MIEL NARANJO DOSIF.: 3.85 €
  ✅ Datos fusionados en memoria (se 